# Pentagonal Evaluation Model

## Import libraries

In [1]:
# Ensure that no .pyc files are generated
import sys

sys.dont_write_bytecode = True

In [2]:
import warnings

import pandas as pd
from kloppy.domain import (
    CardType,
    DuelResult,
    DuelType,
    EventDataset,
    EventType,
    PassResult,
    PassType,
    ShotResult,
    TakeOnResult,
)

from config import players, tournaments

In [3]:
warnings.filterwarnings(
    "ignore",
    message="The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    message="Boolean Series key will be reindexed to match DataFrame index.",
    category=UserWarning,
)

## Load providers data

In [4]:
# StatsBomb match ID for UEFA Euro 2024 Final
match_ids = tournaments.get_all_match_ids(tournaments.EURO_2024)
EURO_2024_FINAL_MATCH_ID = match_ids[0]

In [5]:
# Load players info and team minutes
players_info_df = players.get_players_info(EURO_2024_FINAL_MATCH_ID)

In [6]:
# Load match data
dataset, match_events_df = players.load_match_data(EURO_2024_FINAL_MATCH_ID)

## Define coefficients

In [7]:
VARIABLES_COEFFICIENTS: dict[str, list[float]] = {
    "goals": [0.299, 0.004, 0.098, -0.115, -0.028],
    "assists": [0.044, 0.041, -0.026, 0.208, -0.019],
    "shots": [0.277, 0.008, 0.060, -0.065, -0.016],
    "shots_on_target": [0.295, 0.019, 0.067, -0.087, -0.034],
    "dribbles": [0.107, -0.056, 0.029, 0.124, 0.010],
    "key_passes": [0.011, 0.058, -0.073, 0.299, -0.016],
    "crosses": [-0.161, -0.109, 0.017, 0.438, -0.049],
    "passes": [0.039, 0.119, 0.174, 0.096, -0.024],
    "passing_accuracy": [-0.021, 0.397, -0.149, -0.055, -0.011],
    "short_passes": [0.043, 0.141, 0.141, 0.095, -0.009],
    "short_passing_accuracy": [-0.043, 0.320, -0.059, -0.044, -0.045],
    "long_passes": [0.000, -0.068, 0.345, 0.076, -0.116],
    "long_passing_accuracy": [0.100, 0.331, -0.204, -0.060, 0.050],
    "balls_recovered": [-0.103, -0.010, -0.19, 0.106, 0.308],
    "tackles": [-0.032, -0.040, 0.154, -0.005, 0.164],
    "clearances": [0.112, -0.106, 0.417, -0.207, -0.132],
    "fouls_committed": [0.116, -0.081, -0.043, -0.125, 0.471],
    "yellow_cards": [-0.042, 0.039, -0.194, -0.032, 0.505],
    "red_cards": [0.090, -0.164, 0.336, -0.071, -0.098],
}

In [8]:
FACTOR_COEFFICIENTS: dict[str, float] = {
    "F1": 0.2688,
    "F2": 0.2042,
    "F3": 0.2040,
    "F4": 0.1854,
    "F5": 0.1374,
}

In [9]:
FACTOR_VARIABLES_DF = pd.DataFrame.from_dict(
    VARIABLES_COEFFICIENTS,
    orient="index",
    columns=list(FACTOR_COEFFICIENTS.keys()),
)

## Extract metrics

In [10]:
def extract_player_metrics(
    dataset: EventDataset,
    match_events_df: pd.DataFrame,
    players_info_df: pd.DataFrame,
) -> dict[str, dict[str, str | int]]:
    """
    Extract player metrics needed for Pentagonal Evaluation Model calculation.

    Args:
        dataset: The dataset containing match events and player information.
        match_events_df: DataFrame containing match events.
        players_info_df: DataFrame containing player information.

    Returns:
        A dictionary mapping player nicknames to their metrics.
    """
    player_metrics = {}

    # Map player names to their teams and minutes played for quick access
    nickname_mapping = dict(zip(players_info_df["player_name"], players_info_df["nickname"]))
    team_mapping = dict(zip(players_info_df["player_name"], players_info_df["team_name"]))
    minutes_mapping = dict(zip(players_info_df["player_name"], players_info_df["minutes_played"]))

    # Get player position mapping and assists count
    position_mapping = players.get_players_position_group(dataset)

    for player_name in match_events_df["player"].unique():
        # Get specific player event data
        player_df = match_events_df.loc[match_events_df["player"] == player_name]
        player_dataset = dataset.filter(
            lambda event, player_name=player_name: (
                False if not hasattr(event.player, "name") else event.player.name == player_name
            )
        )

        # Use nickname, team and minutes mappings
        player_nickname = nickname_mapping[player_name] or player_name
        player_team = team_mapping[player_name]
        player_minutes = minutes_mapping[player_name]

        # Attacking metrics
        goals = len(player_df.loc[(player_df["event_type"] == "SHOT") & (player_df["result"] == "GOAL")])
        assists = players.calculate_metric_for_player_dataset(
            player_dataset,
            EventType.PASS,
            result_filter=PassResult.COMPLETE,
            qualifier_filter=PassType.ASSIST,
        )
        shots = len(player_df.loc[player_df["event_type"] == "SHOT"])
        shots_on_target = len(
            player_df.loc[
                (player_df["event_type"] == "SHOT") & (player_df["result"].isin(["GOAL", "BLOCKED", "SAVED"]))
            ]
        )
        dribbles = len(player_df.loc[(player_df["event_type"] == "TAKE_ON") & (player_df["success"])])
        key_passes = players.calculate_metric_for_player_dataset(
            player_dataset,
            EventType.PASS,
            result_filter=PassResult.COMPLETE,
            qualifier_filter=PassType.SHOT_ASSIST,
        )
        crosses = len(
            player_df.loc[
                (player_df["event_type"] == "PASS") & (player_df["pass_type"] == "CROSS") & (player_df["success"])
            ]
        )

        # Distribution metrics
        passes = len(player_df.loc[(player_df["event_type"] == "PASS") & (player_df["success"])])
        passing_accuracy = (
            (passes / len(player_df.loc[player_df["event_type"] == "PASS"]))
            if len(player_df.loc[player_df["event_type"] == "PASS"]) > 0
            else 0.0
        )
        short_passes = len(
            player_df.loc[
                (player_df["event_type"] == "PASS") & (player_df["pass_type"] == "SIMPLE_PASS") & (player_df["success"])
            ]
        )
        short_passing_accuracy = (
            (
                short_passes
                / len(player_df.loc[(player_df["event_type"] == "PASS") & (player_df["pass_type"] == "SIMPLE_PASS")])
            )
            if len(player_df.loc[(player_df["event_type"] == "PASS") & (player_df["pass_type"] == "SIMPLE_PASS")]) > 0
            else 0.0
        )
        long_passes = len(
            player_df.loc[
                (player_df["event_type"] == "PASS") & (player_df["pass_type"] == "LONG_BALL") & (player_df["success"])
            ]
        )
        long_passing_accuracy = (
            (
                long_passes
                / len(player_df.loc[(player_df["event_type"] == "PASS") & (player_df["pass_type"] == "LONG_BALL")])
            )
            if len(player_df.loc[(player_df["event_type"] == "PASS") & (player_df["pass_type"] == "LONG_BALL")]) > 0
            else 0.0
        )

        # Defensive metrics
        balls_recovered = len(player_df.loc[(player_df["event_type"] == "RECOVERY")])
        tackles = len(
            player_df.loc[
                (player_df["event_type"] == "DUEL")
                & (player_df["duel_type"] == "SLIDING_TACKLE")
                & (player_df["success"])
            ]
        )
        clearances = len(player_df.loc[(player_df["event_type"] == "CLEARANCE")])
        fouls_committed = len(player_df.loc[(player_df["event_type"] == "FOUL_COMMITED")])
        yellow_cards = len(
            player_df.loc[(player_df["event_type"] == "CARD") & (player_df["card_type"] == "FIRST_YELLOW")]
        )
        red_cards = len(
            player_df.loc[(player_df["event_type"] == "CARD") & (player_df["card_type"].isin(["SECOND_YELLOW", "RED"]))]
        )

        player_metrics[player_nickname] = {
            "team": player_team,
            "position": position_mapping[player_name],
            "minutes_played": player_minutes,
            "goals": goals,
            "assists": assists,
            "shots": shots,
            "shots_on_target": shots_on_target,
            "dribbles": dribbles,
            "key_passes": key_passes,
            "crosses": crosses,
            "passes": passes,
            "passing_accuracy": passing_accuracy,
            "short_passes": short_passes,
            "short_passing_accuracy": short_passing_accuracy,
            "long_passes": long_passes,
            "long_passing_accuracy": long_passing_accuracy,
            "balls_recovered": balls_recovered,
            "tackles": tackles,
            "clearances": clearances,
            "fouls_committed": fouls_committed,
            "yellow_cards": yellow_cards,
            "red_cards": red_cards,
        }

    return player_metrics

In [11]:
def extract_player_metrics_v2(
    dataset: EventDataset,
    players_info_df: pd.DataFrame,
) -> dict[str, dict[str, str | int]]:
    """
    Extract player metrics needed for Pentagonal Evaluation Model calculation.

    Args:
        dataset: The dataset containing match events and player information.
        players_info_df: DataFrame containing player information.

    Returns:
        A dictionary mapping player nicknames to their metrics.
    """
    player_metrics: dict[str, dict] = {}

    # Get player position mapping and assists count
    players_position_dict = players.get_players_position_group(dataset)

    for idx, player_name in enumerate(players_info_df["player_name"]):
        # Get specific player event data
        player_dataset = dataset.filter(
            lambda event, player_name=player_name: (
                False if not hasattr(event.player, "name") else event.player.name == player_name
            )
        )

        # Get nickname, team and minutes from players_info_df
        player_nickname = players_info_df.at[idx, "nickname"] or player_name
        player_team = players_info_df.at[idx, "team_name"]
        player_minutes = players_info_df.at[idx, "minutes_played"]

        # Attacking metrics
        goals = players.calculate_metric_for_player_dataset(player_dataset, EventType.SHOT, ShotResult.GOAL)
        assists = players.calculate_metric_for_player_dataset(
            player_dataset,
            EventType.PASS,
            PassResult.COMPLETE,
            PassType.ASSIST,
        )
        shots = players.calculate_metric_for_player_dataset(player_dataset, EventType.SHOT)
        shots_on_target = players.calculate_metric_for_player_dataset(
            player_dataset,
            EventType.SHOT,
            [ShotResult.GOAL, ShotResult.BLOCKED, ShotResult.SAVED],
        )
        dribbles = players.calculate_metric_for_player_dataset(player_dataset, EventType.TAKE_ON, TakeOnResult.COMPLETE)
        key_passes = players.calculate_metric_for_player_dataset(
            player_dataset,
            EventType.PASS,
            PassResult.COMPLETE,
            PassType.SHOT_ASSIST,
        )
        crosses = players.calculate_metric_for_player_dataset(
            player_dataset,
            EventType.PASS,
            PassResult.COMPLETE,
            PassType.CROSS,
        )

        # Distribution metrics
        passes = players.calculate_metric_for_player_dataset(player_dataset, EventType.PASS, PassResult.COMPLETE)
        passing_accuracy = (
            passes / players.calculate_metric_for_player_dataset(player_dataset, EventType.PASS)
            if players.calculate_metric_for_player_dataset(player_dataset, EventType.PASS) > 0
            else 0.0
        )
        short_passes = players.calculate_metric_for_player_dataset(
            player_dataset,
            EventType.PASS,
            PassResult.COMPLETE,
            PassType.LONG_BALL,
            exclude_qualifier=True,
        )
        short_passing_accuracy = (
            (
                short_passes
                / players.calculate_metric_for_player_dataset(
                    player_dataset,
                    EventType.PASS,
                    qualifier_filter=PassType.LONG_BALL,
                    exclude_qualifier=True,
                )
            )
            if players.calculate_metric_for_player_dataset(
                player_dataset,
                EventType.PASS,
                qualifier_filter=PassType.LONG_BALL,
                exclude_qualifier=True,
            )
            > 0
            else 0.0
        )
        long_passes = players.calculate_metric_for_player_dataset(
            player_dataset,
            EventType.PASS,
            PassResult.COMPLETE,
            PassType.LONG_BALL,
        )
        long_passing_accuracy = (
            (
                long_passes
                / players.calculate_metric_for_player_dataset(
                    player_dataset,
                    EventType.PASS,
                    qualifier_filter=PassType.LONG_BALL,
                )
            )
            if players.calculate_metric_for_player_dataset(
                player_dataset,
                EventType.PASS,
                qualifier_filter=PassType.LONG_BALL,
            )
            > 0
            else 0.0
        )

        # Defensive metrics
        balls_recovered = players.calculate_metric_for_player_dataset(player_dataset, EventType.RECOVERY)
        tackles = players.calculate_metric_for_player_dataset(
            player_dataset,
            EventType.DUEL,
            DuelResult.WON,
            [DuelType.GROUND, DuelType.SLIDING_TACKLE],
        )
        clearances = players.calculate_metric_for_player_dataset(player_dataset, EventType.CLEARANCE)
        fouls_committed = players.calculate_metric_for_player_dataset(player_dataset, EventType.FOUL_COMMITTED)
        yellow_cards = players.calculate_metric_for_player_dataset(
            player_dataset,
            EventType.CARD,
            card_filter=CardType.FIRST_YELLOW,
        )
        red_cards = players.calculate_metric_for_player_dataset(
            player_dataset,
            EventType.CARD,
            card_filter=[CardType.SECOND_YELLOW, CardType.RED],
        )

        player_metrics[player_nickname] = {
            "team": player_team,
            "position": players_position_dict[player_name],
            "minutes_played": player_minutes,
            "goals": goals,
            "assists": assists,
            "shots": shots,
            "shots_on_target": shots_on_target,
            "dribbles": dribbles,
            "key_passes": key_passes,
            "crosses": crosses,
            "passes": passes,
            "passing_accuracy": passing_accuracy,
            "short_passes": short_passes,
            "short_passing_accuracy": short_passing_accuracy,
            "long_passes": long_passes,
            "long_passing_accuracy": long_passing_accuracy,
            "balls_recovered": balls_recovered,
            "tackles": tackles,
            "clearances": clearances,
            "fouls_committed": fouls_committed,
            "yellow_cards": yellow_cards,
            "red_cards": red_cards,
        }

    return player_metrics

In [12]:
# Extract player metrics
player_metrics = extract_player_metrics_v2(dataset, players_info_df)

## Calculate the Pentagonal Evaluation Metric

In [13]:
def calculate_players_pentagonal_score(player_metrics: dict, FACTOR_VARIABLES_DF: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate Pentagonal Score for each player.

    Args:
        player_metrics: A dictionary containing player metrics.
        FACTOR_VARIABLES_DF: A DataFrame containing the factor variables for the Pentagonal Score calculation.

    Returns:
        A DataFrame containing Pentagonal Scores for each player.
    """

    # Initialize list to hold players Pentagonal Scores
    pentagonal_scores_list = []

    for player, metrics in player_metrics.items():
        player_points = 0
        factor_scores = {}

        for factor in FACTOR_VARIABLES_DF.columns:
            factor_score = sum(
                metrics[metric] * FACTOR_VARIABLES_DF.loc[metric, factor] for metric in FACTOR_VARIABLES_DF.index
            )
            factor_scores[factor] = factor_score
            player_points += factor_score * FACTOR_COEFFICIENTS[factor]

        # Append player Pentagonal Scores to list
        pentagonal_scores_list.append(
            {
                "player": player,
                "team": metrics["team"],
                "position": metrics["position"],
                "pentagonal_score": player_points,
                "f1": factor_scores["F1"],
                "f2": factor_scores["F2"],
                "f3": factor_scores["F3"],
                "f4": factor_scores["F4"],
                "f5": factor_scores["F5"],
                "minutes_played": metrics["minutes_played"],
                "goals": metrics["goals"],
                "assists": metrics["assists"],
                "shots": metrics["shots"],
                "shots_on_target": metrics["shots_on_target"],
                "dribbles": metrics["dribbles"],
                "key_passes": metrics["key_passes"],
                "crosses": metrics["crosses"],
                "passes": metrics["passes"],
                "passing_accuracy": metrics["passing_accuracy"],
                "short_passes": metrics["short_passes"],
                "short_passing_accuracy": metrics["short_passing_accuracy"],
                "long_passes": metrics["long_passes"],
                "long_passing_accuracy": metrics["long_passing_accuracy"],
                "balls_recovered": metrics["balls_recovered"],
                "tackles": metrics["tackles"],
                "clearances": metrics["clearances"],
                "fouls_committed": metrics["fouls_committed"],
                "yellow_cards": metrics["yellow_cards"],
                "red_cards": metrics["red_cards"],
            }
        )

    # Convert list to DataFrame
    pentagonal_scores_df = pd.DataFrame(pentagonal_scores_list)

    # Sort by Pentagonal Scores and rank players
    pentagonal_scores_df = pentagonal_scores_df.sort_values("pentagonal_score", ascending=False).reset_index(drop=True)
    pentagonal_scores_df.insert(0, "rank", range(1, len(pentagonal_scores_df) + 1))

    return pentagonal_scores_df

In [14]:
# Calculate Pentagonal Scores
pentagonal_scores_df = calculate_players_pentagonal_score(player_metrics, FACTOR_VARIABLES_DF)

In [15]:
# Display all players ranked by Pentagonal Scores
display(pentagonal_scores_df)

,rank,player,team,position,pentagonal_score,f1,f2,f3,f4,f5,...,short_passes,short_passing_accuracy,long_passes,long_passing_accuracy,balls_recovered,tackles,clearances,fouls_committed,yellow_cards,red_cards
0,1,Aymeric Laporte,Spain,Defender,13.969456,7.619209,20.000172,28.704918,13.544131,-3.853767,...,75,0.974026,5,0.833333,1,0,7,0,0,0
1,2,Robin Le Normand,Spain,Defender,13.759490,6.396544,21.030445,25.197242,15.269176,-1.640907,...,78,0.987342,2,0.400000,4,1,1,0,0,0
2,3,Daniel Carvajal,Spain,Defender,10.965598,4.584029,15.285061,21.137119,12.903368,-0.669979,...,56,0.875000,7,0.777778,5,3,2,1,0,0
3,4,Fabián Ruiz,Spain,Midfielder,9.879686,4.597148,14.364588,17.355746,11.701185,0.005473,...,51,0.879310,5,0.833333,7,2,0,0,0,0
4,5,Marc Cucurella,Spain,Defender,7.664846,3.993512,11.015148,14.447870,7.689327,-0.224673,...,41,0.976190,1,1.000000,1,1,4,3,0,0
5,6,Nico Williams,Spain,Midfielder,7.620867,4.838618,10.241939,12.854320,8.010447,0.883732,...,36,0.837209,5,0.625000,5,0,0,3,0,0
6,7,Kyle Walker,England,Defender,6.215719,2.859687,8.586886,13.239573,6.443607,-1.469616,...,30,0.937500,6,0.600000,1,2,2,0,0,0
7,8,Declan Rice,England,Midfielder,5.874226,3.470026,8.419425,9.300604,6.292377,1.152158,...,29,0.743590,3,1.000000,7,0,0,1,0,0
8,9,Unai Simón,Spain,Goalkeeper,5.736019,2.116483,8.930783,10.118350,6.856250,0.059317,...,31,1.000000,3,0.333333,5,0,0,0,0,0
9,10,John Stones,England,Defender,5.373815,3.006440,6.263386,13.616206,3.755377,-1.362829,...,23,0.920000,7,0.700000,1,1,8,1,1,0
